In [1]:
pc = 'FotR_ex'
import os
config = {
    'root_dir':{
        'laptop': '/home/migueljaraiz/anaconda3/repos/',
        'cluster': '/home/m.jaraiz/Documentos/DATASETS/data_TIFON/rans3_extended/outputs/',
        'pc_pro': '/home/ninjaraiz/anaconda3/repos/'
    },
    'folder_to_save':{
        'laptop': '/home/migueljaraiz/anaconda3/repos/GMM_TIFON/',
        'cluster': '/home/m.jaraiz/Documentos/GMM/GMM_TIFON/',
        'FotR_ex': './example_GMM/'
    },
    'pylom_path': {
        'laptop': '/home/migueljaraiz/anaconda3/repos/pyLowOrder',
        'cluster': '/home/m.jaraiz/repos/pyLowOrder',
        'pc_pro': '/home/ninjaraiz/anaconda3/repos/pyLowOrder'
    }   
}

for key in ['root_dir', 'pylom_path']:
    config[key]['FotR_ex'] = [path for _, path in config[key].items()]


import sys
try:
    import pyLOM
    print('Entorno con pyLOM instalado')
except ImportError as e:
    print(e)
    print('Imported by local repository')

    if pc == 'FotR_ex':
        for path in config['pylom_path'][pc]:
            if os.path.exists(path):
                sys.path.append(path)
                print(f'Found pylom path: {path}')
                break
            print('Not found pylom path')
    else:
        sys.path.append(config['pylom_path'][pc])

from FotR import FRODO
import matplotlib.pyplot as plt

import numpy as np, torch

def symlog(x, linthresh=1e-3):
    if isinstance(x, torch.Tensor):
        return torch.sign(x) * torch.log10(1 + torch.abs(x) / linthresh)
    elif isinstance(x, np.ndarray):
        return np.sign(x) * np.log1p(np.abs(x) / linthresh)

for folder_path in config['root_dir'][pc]:
    if os.path.exists(folder_path):
        config['root_dir'][pc] = folder_path
        print('Folder root_dir found succesfull.')
        break
if isinstance(config['root_dir'][pc], list):
    raise LookupError('Folder root_dir not found with this configuration.')
    
db = FRODO(
    root_dir = config['root_dir'][pc],
    format = 'PYLOM',
    file = 'CADGroup_3_completo_stage_1.h5')

db.extract_inputs(
    keys_inputs={
        'ptos': 'xyz',    # coordenadas del mallado → data_dict['inputs']['ptos']
        'aoa': 'aoa',   # variable paramétrica   → data_dict['inputs']['aoa']
        'mach': 'M',   # variable paramétrica   → data_dict['inputs']['M']
    },
    keys_aux={},
)

db.extract_outputs(
    keys_outputs={
        'cp': 'BoundaryValues_CoefPressure',
        'cf': 'BoundaryValues_CoefSkinFrictionTangential',
        'rhoV': 'State_Momentum_interp',
        # 'rho': 'State_Density_interp'
        }
    
)

No module named 'pyLOM'
Imported by local repository
Not found pylom path
Found pylom path: /home/m.jaraiz/repos/pyLowOrder


0 Warning! Import - NVTX not present!


Folder root_dir found succesfull.


[PYLOMReader] Dataset loaded in 2.387 s
Dataset of 9 variables:
  > CD  - max = 0.0853571747219292, min = 0.0211941016109823
  > CL  - max = 0.7128004506389523, min = 0.0017507663531984
  > CMy  - max = 0.0116312300308043, min = -0.0814396710024715
  > M  - max = 1.489529013633728, min = 0.3043437898159027
  > Re  - max = 2127846.6018327894, min = 315933.6875
  > aoa  - max = 4.983091831207275, min = 0.0221543479710817
  > varCD  - max = 0.0058622738263488, min = 1.5518123489720028e-18
  > varCL  - max = 0.0124718994142255, min = 5.734072955026623e-15
  > varCM  - max = 0.001136587303561, min = 1.5261117831895702e-16
and 29 fields with 13862 points:
  > AugStateGrad_DensityGradient_interp - max = 1625.1697879448745, min = -7090.855791525813, avg = -1.0555945956806825
  > AugStateGrad_EnergyStagnationDensityGradient_interp - max = 7441.320562346408, min = -28504.520387444205, avg = -1.8056840656202633
  > AugStateGrad_MomentumXGradient_interp - max = 

In [2]:
# Acceder a los datos
xyz = db.sets.get_xyz()           # (npoints, 3)
aoa   = db.sets.get_variable('aoa') # (500,)
mach = db.sets.get_variable('mach')   # (500,)
cp  = db.sets.get_field('cp')     # (npoints, 500)
cf = db.sets.get_field('cf')     # (npoints, 500)


# T = db.sets.get_field('T')
# rho = db.sets.get_field('rho')

# gradrhox = db.sets.get_field('gradrho')[0, :, :]#np.linalg.norm(db.sets.get_field('gradrho'), ord=2,axis=0)
# gradTx = db.sets.get_field('gradT')[0, :, :]#np.linalg.norm(db.sets.get_field('gradT'), ord=2,axis=0)

rhoV = db.sets.get_field('rhoV') #np.linalg.norm(db.sets.get_field('rhoV'), ord=2, axis=0)


from FotR import SAM
xyz_sort, order_sort = SAM.Weapons.sort_by_centroid(xyz)
cp_sort = cp[order_sort, :]
cf_sort = cf[order_sort, :]
# T_sort = T[order_sort, :]
# rho_sort = rho[order_sort, :]

# gradrhox_sort = gradrhox[order_sort, :]
# gradTx_sort = gradTx[order_sort, :]

rhoV_sort = rhoV[:, order_sort, :]

sep = 1
clusters = 2

In [ ]:
stencil = 310

polyorder = 3

# ── derivada por longitud de arco ─────────────────────────────────────────
dcp_ds = torch.zeros(cp_sort.shape, dtype=torch.float64)
dcp2_ds = torch.zeros(cp_sort.shape, dtype=torch.float64)
dcf_ds = torch.zeros(cf_sort.shape, dtype=torch.float64)
dcf2_ds = torch.zeros(cf_sort.shape, dtype=torch.float64)

for case in range(cp_sort.shape[1]):
    dcp_ds[:, case] = SAM.Weapons.surface_derivative(
        X=torch.from_numpy(xyz_sort),
        f=torch.from_numpy(cp_sort)[:, case],
        order=1,
        stencil_width=stencil,   
        poly_order=polyorder,
    )
    
    dcf_ds[:, case] = SAM.Weapons.surface_derivative(
        X=torch.from_numpy(xyz_sort),
        f=torch.from_numpy(cf_sort)[:, case],
        order=1,
        stencil_width=stencil,   
        poly_order=polyorder,
    )
    dcp2_ds[:, case] = SAM.Weapons.surface_derivative(
        X=torch.from_numpy(xyz_sort),
        f=dcp_ds[:, case],
        order=1,
        stencil_width=stencil,
        poly_order=polyorder,
    )

    dcf2_ds[:, case] = SAM.Weapons.surface_derivative(
        X=torch.from_numpy(xyz_sort),
        f=dcf_ds[:, case],
        order=1,
        stencil_width=stencil,
        poly_order=polyorder,
    )

In [4]:
div_rhoV = SAM.DifferentialOperators.divergence(
    xyz_sort,
    rhoV_sort[:, :, 0:5],
    stencil_width=310,
    poly_order=2
)

In [ ]:
scale_log = True
dcp_ds_log = symlog(cp_sort, linthresh=1e-4) if scale_log else dcp_ds
dcp2_ds_log = symlog(dcp2_ds, linthresh=1e-4) if scale_log else dcp2_ds

dcf2_ds_log = symlog(dcf2_ds, linthresh=1e-4) if scale_log else dcf2_ds
dcf_ds_log = symlog(dcf_ds, linthresh=1e-4) if scale_log else dcf_ds

db_one = db.copy()
db_one.sets.add_aux(
    array_name = 'dcp_ds_log',
    array = dcp_ds_log.numpy(),
    notes = 'Log dcp_ds')

db_one.sets.add_aux(
    array_name = 'dcp2_ds_log',
    array = dcp2_ds_log.numpy(),
    notes = 'Log dcp2_ds')

# db_one.sets.add_aux(
#     array_name = 'gradrhox_log',
#     array = gradrhox_log.numpy(),
#     notes = 'Log gradrhox'
# )

# db_one.sets.add_aux(
#     array_name = 'gradTx_log',
#     array = gradTx_log.numpy(),
#     notes = 'Log gradTx'
# )

db_one.data_dict['inputs']['ptos'] = tensor_ptos.numpy()
db_one.data_dict['outputs']['cp'] = tensor_cp_filtered.numpy()
[db_one.data_dict['outputs'].pop(key, None) for key in ['gradT', 'gradrho']]
db_one.sets.create_jset(verbose=False)
# display(db_one.df_data)

db_one.sets.create_jset(verbose=False)

features = ['dcp_ds_log', 'dcp2_ds_log'] # , 'gradrhox_log', 'gradTx_log'
folder_name = '_'.join(features)
df_data_complete, _ = SAM.Weapons.GMM(
    df_data=db_one.df_data,
    BIC_study=False,
    groupby=["aoa", "mach"],
    nclusters=clusters,
    features=features,
    save_pictures=True,
    folder_to_save=os.path.join(config['folder_to_save'][pc], f'{folder_name}/sep_{sep}/c_{clusters}'),
    n_components_range=range(1, 5),
    covariance_type="diag",
    max_iter=300,
    random_state=42,
    return_metrics_table=True,
    plot_global_analysis=True,
    verbose = True
)

os.makedirs(os.path.join(config['folder_to_save'][pc], f'{folder_name}/sep_{sep}/c_{clusters}/isolate_cases'), exist_ok=True)
for case in range(len(df_data_complete['aoa'].unique())):
    df_case = df_data_complete.groupby(['aoa', 'mach']).get_group((df_data_complete['aoa'].unique()[case], df_data_complete['mach'].unique()[case]))
    fig, ax = plt.subplots(1, 1, figsize=(10, 6))
    
    ax.scatter(df_case['x'], df_case['z'], c='black', s=1, edgecolor='k', alpha=0.7)
    ax.set_xlabel('x')
    ax.set_ylabel('z')
    ax.set_title(f'GMM Clustering (case {case}) - features: {features} - sep: {sep}')
    ax.set_ylim(df_case['z'].min() - 0.1, df_case['z'].max() + 0.1)
    
    ax1 = ax.twinx()
    ax1.scatter(df_case['x'], df_case['cp'], c=df_case['clusters_GMM'], s=20, alpha=0.7, cmap='viridis')
    ax1.set_ylabel('cp')
    
    plt.savefig(os.path.join(config['folder_to_save'][pc], f'{folder_name}/sep_{sep}/c_{clusters}/isolate_cases', f'case_{case}.png'))
    plt.close()

In [ ]:
case = 20
scale = 7
markersize_dcp = 1
def get_column_from_df(column, case, df):
    if isinstance(column, (list, tuple)):
        lista = []
        for col in column:
            lista.append(df.groupby(['aoa', 'mach']).get_group((df['aoa'].unique()[case], df['mach'].unique()[case]))[col])
        return lista
    
    if isinstance(column, str):
        serie = df_data_complete.groupby(['aoa', 'mach']).get_group((df_data_complete['aoa'].unique()[case], df_data_complete['mach'].unique()[case]))[column]
        return serie
    
[x, z, cp, clusters] = get_column_from_df(['x', 'z', 'cp', 'clusters_GMM'], case, df_data_complete)

fig, ax = plt.subplots(2, 1, figsize=(12, 2*6))
# ax = ax.flatten()
ax[0].scatter(
    x, z,
    c='black', s=1
)
ax00 = ax[0].twinx()
ax00.scatter(
    x, dcp_ds_filtered[:, case], c='blue', s=markersize_dcp
)

ax01 = ax[0].twinx()
ax01.scatter(
    x, dcp2_ds[:, case], c='red', s=markersize_dcp
)
ax[0].set_ylim(bottom = z.min()*scale, top = z.max()*scale)
# Poner un tercer eje a la izquierda con cp
ax_cp = ax[0].twinx()
ax_cp.scatter(
    x, tensor_cp_filtered[:, case], c='green', s=markersize_dcp
)
ax_cp.spines['left'].set_position(('outward', 60))
ax_cp.spines['left'].set_color('green')
ax_cp.tick_params(axis='y', colors='green')
ax_cp.invert_yaxis()
# Arreglar ejes de los twinx
ax00.set_yscale('log')
ax01.set_yscale('log')
#separar ejes secundarios y poner del mismo color que los puntos
ax00.spines['right'].set_position(('outward', 60))
ax01.spines['right'].set_position(('outward', 120))
ax00.spines['right'].set_color('blue')
ax01.spines['right'].set_color('red')
ax00.tick_params(axis='y', colors='blue')
ax01.tick_params(axis='y', colors='red')
ax[0].set_xlabel('x')
ax[0].set_ylabel('z')
ax00.set_ylabel('dcp/ds', color='blue')
ax01.set_ylabel('d2cp/ds2', color='red')
ax[0].set_title(f'Case {case}')

ax[1].scatter(
    x, z, c='black', s=markersize_dcp, alpha=0.7)
ax[1].set_xlabel('x')
ax[1].set_ylabel('z')
ax[1].set_ylim(z.min() - 0.1, z.max() + 0.1)
ax10 = ax[1].twinx()
ax10.scatter(
    x, cp, c=clusters, s=markersize_dcp, alpha=0.7, cmap='viridis')
ax10.set_ylabel('cP')
ax10.invert_yaxis()

fig.suptitle(f'Case {case} - features: {features} - sep: {sep}')
plt.show()


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10*2, 6))
ax = ax.flatten()
sc0 = ax[0].scatter(
    x, z, c='black', s=1, edgecolor='k', alpha=0.7)
ax[0].set_xlabel('x')
ax[0].set_ylabel('z')
ax[0].set_title(f'GMM Clustering (case {case})')
ax[0].set_ylim(z.min() - 0.1, z.max() + 0.1)
ax0 = ax[0].twinx()
sc01 = ax0.scatter(
    x, cp, c=clusters, s=50, edgecolor='k', alpha=0.7, cmap='viridis')
ax0.set_ylabel('dcp_ds')

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 1, figsize=(8, 6))
ax.scatter(
    df_data_complete['dcp_ds_log'],
    df_data_complete['gradrhox_log'],
    c=df_data_complete['clusters_GMM'],
    cmap='viridis',
    label='All data')
ax.set_xlabel('dcp_ds')
ax.set_xscale('symlog')
ax.set_ylabel('gradrho')
ax.set_yscale('symlog')


import os
os.makedirs('/home/m.jaraiz/Documentos/GMM/GMM_TIFON/clusters_isolate/', exist_ok=True)
ngroups = len(df_data_complete.groupby(['aoa', 'mach']))

nrows = 3
ncols = ngroups // nrows + (ngroups % nrows > 0)
fig, ax = plt.subplots(nrows, ncols, figsize=(8*ncols, 6*nrows))
ax = ax.flatten()
for i, (aoa, mach) in enumerate(df_data_complete.groupby(['aoa', 'mach']).groups.keys()):
    group = df_data_complete[(df_data_complete['aoa'] == aoa) & (df_data_complete['mach'] == mach)]
    ax[i].scatter(
        group['x'],
        group['z'],
        c=group['clusters_GMM'],
        cmap='viridis',
        label=f'aoa={aoa}, M={mach}'
    )
    ax1 = ax[i].twinx()
    ax1.scatter(
        group['x'],
        group['cp'],
        c='red'
    )
    ax1.set_ylabel('cp', color='red')
    # for color, cluster in zip(['yellow', 'blue'], group['clusters_GMM'].unique()):
    #     cluster_group = group[group['clusters_GMM'] == cluster]
    #     ax[i].scatter(cluster_group['dcp_ds'], cluster_group['gradrho'], c=color)
    ax[i].set_xlabel('dcp_ds')
    ax[i].set_ylabel('gradrho')
    ax[i].set_title(f'aoa={aoa}, M={mach}')
    # ax[i].legend()
    # plt.savefig(f'/home/m.jaraiz/Documentos/GMM/GMM_TIFON/clusters_isolate/cluster_aoa_{aoa}_mach_{mach}.png')

# for key, group in df_data_complete.groupby(['aoa', 'mach']):
#     ax.scatter(group['dcp_ds'], group['gradrho'], label=f'aoa={key[0]}, M={key[1]}')
# ax.set_xlabel('dcp_ds')
# ax.set_ylabel('gradrho')
# ax.legend()
# plt.title(f'Scatter plot of dcp_ds vs gradrho (sep={sep})')
# plt.show()

#### Calculation of derivates in isolated case

In [ ]:
import matplotlib.pyplot as plt
import io
from PIL import Image


scale = 0.7
case_gif = 27

# ── Precomputar límites globales para ejes fijos ──────────────────────────────
# Se usan el subconjunto más denso (sep mínimo) como referencia
_ref_ptos = torch.from_numpy(xyz_sort[::20, :])
_ref_cp   = torch.from_numpy(cp_sort[::20, :])
_ref_dcp  = SAM.Weapons.surface_derivative(X=_ref_ptos, f=_ref_cp[:, case_gif], order=1)

Y_CP_MIN  = float(_ref_cp[:, case_gif].min())
Y_CP_MAX  = float(_ref_cp[:, case_gif].max())
Y_DCP_MIN = float(_ref_dcp.min())
Y_DCP_MAX = float(_ref_dcp.max())
X_MIN     = float(_ref_ptos[:, 0].min())
X_MAX     = float(_ref_ptos[:, 0].max())
Y_GEO_MIN = float(_ref_ptos.min()) * scale
Y_GEO_MAX = float(_ref_ptos.max()) * scale
# ─────────────────────────────────────────────────────────────────────────────

frames: list[Image.Image] = []

for sep in range(220, 20, -2):
    tensor_ptos = torch.from_numpy(xyz_sort[::sep, :])
    tensor_cp   = torch.from_numpy(cp_sort[::sep,  :])
    tensor_gradrho = torch.from_numpy(gradrhox_sort[::sep, :])
    tensor_gradT = torch.from_numpy(gradTx_sort[::sep, :])
    N = tensor_ptos.shape[0]

    # ── derivada por longitud de arco ─────────────────────────────────────────
    dcp_ds = torch.zeros(tensor_cp.shape, dtype=torch.float64)
    for case in range(tensor_cp.shape[1]):
        dcp_ds[:, case] = SAM.Weapons.surface_derivative(
            X=tensor_ptos, f=tensor_cp[:, case], order=1
        )
    # ─────────────────────────────────────────────────────────────────────────

    fig, ax = plt.subplots(1, 1, figsize=(12, 6))
    fig.patch.set_facecolor('white')

    pl = ax.plot(
        tensor_ptos[:, 0],
        tensor_cp[:, case_gif],
        c = 'black',
        linewidth = 1,
    )
    ax.set_ylabel('y', fontsize=11)
    ax.set_xlabel('x', fontsize=11)
    ax.set_xlim(X_MIN, X_MAX)
    ax.set_ylim(Y_CP_MIN - 0.05 * abs(Y_CP_MIN), Y_CP_MAX + 0.05 * abs(Y_CP_MAX))
    ax.set_yticks([])

    # --- dcp/ds (eje derecho principal) --------------------------------------
    ax2 = ax.twinx()
    ax2.spines['right'].set_position(('outward', 0))   # superpuesto a ax1

    y  = tensor_gradrho[:, case_gif]
    n  = len(y)
    # mitad izquierda → rojo  |  mitad derecha → azul  (cara presión / succión)
    color = ['red', 'blue']

    sc2 = ax2.scatter(
        x=tensor_ptos[:n//2, 0],
        y=y[:n//2],
        c=[color[0]] * (n // 2),
        s=15,
        alpha=0.8,zorder=4,
    )
    
    sc3 = ax2.scatter(
        x=tensor_ptos[n - n // 2:, 0],
        y=y[n - n // 2:],
        c=[color[1]] * (n // 2),
        s=15,
        alpha=0.8,
        zorder=4,
    )
    ax2.axhline(0, color='k', lw=0.8, ls='--', alpha=0.4)
    ax2.set_ylabel('grad_rho', fontsize=11)
    ax2.set_yscale('symlog')
    ax2.tick_params(axis='y', colors='crimson')
    ax2.set_ylim(Y_DCP_MIN * 1.1, Y_DCP_MAX * 1.1)
    # ax2.set_yscale('symlog', linthresh=1e-2)

    # --- título con info del frame -------------------------------------------
    ax.set_title(
        f'case {case_gif}  ·  sep = {sep}  ·  N = {N} pts',
        fontsize=12, pad=8
    )
    # Crear leyenda personalizada (linea negra: cp | puntos rojos/azules: dcp/ds). Crear linea negra y marcadores rojos y azules. Tres elementos: cp, dcp_ds_intrados, dcp_ds_extrados
    
    plt.legend(
        handles=[pl[0], sc2, sc3],
        labels=['cp', 'intrados', 'extrados'],
        loc='upper right',
        fontsize=10)
    # fig.legend(
    #     [pl, sc2],
    #     labels=['intrados', 'extrados'], loc='upper right', fontsize=10)
    plt.tight_layout()

    # ── capturar frame en memoria → PIL Image ─────────────────────────────────
    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=100, bbox_inches='tight')
    buf.seek(0)
    frames.append(Image.open(buf).copy())   # .copy() libera el buffer
    plt.close(fig)
    buf.close()
    # ─────────────────────────────────────────────────────────────────────────

# ── Guardar GIF ───────────────────────────────────────────────────────────────
gif_name = f'tensor_gradrho_case{case_gif}_evolution.gif'
gif_path = config[pc]['folder_to_save']

import os
os.makedirs(gif_path, exist_ok=True)

frames[0].save(
    os.path.join(gif_path, gif_name),
    save_all=True,
    append_images=frames[1:],
    duration=200,          # ms por frame  → ajusta a tu gusto
    loop=0,                # 0 = bucle infinito
    optimize=False,
)

print(f'GIF guardado en: {gif_path}  ({len(frames)} frames)')

#### Calculation of clusters depending on the filter in airfoil